# 1. Генерация датасета для Schema Matching и Entity Resolution

Этот пайплайн генерирует единый датасет для обоих модулей:
- **Schema Matching**: JSON с эмбеддингами столбцов и метками `base_name`
- **Entity Resolution**: CSV-пары таблиц с ground truth дубликатами

Процесс:
1. Загрузка конфига
2. Генерация данных (EntityPool → таблицы → LLM описания → эмбеддинги)
3. Визуальный и статистический анализ полученного датасета

## 1.1 Загрузка конфигурации

In [ ]:
import json
import os
import sys
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

CONFIG_PATH = 'config.json'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    full_config = json.load(f)

data_gen_params = full_config['data_generation']
print(f"Ollama host: {data_gen_params['ollama_host']}")
print(f"LLM: {data_gen_params['llm_model']}")
print(f"Embedding: {data_gen_params['embedding_model']}")
print(f"ER пар: train={data_gen_params['num_train_pairs']}, val={data_gen_params['num_val_pairs']}, test={data_gen_params['num_test_pairs']}")
print(f"Extra SM таблиц: {data_gen_params['num_extra_sm_tables']}")

## 1.2 Генерация датасета

In [ ]:
from table_unifier.data_generation import DataGenConfig, UnifiedDatasetGenerator

# Создаём конфиг из словаря
data_gen_config = DataGenConfig(**data_gen_params)

print(f"Всего ER пар: {data_gen_config.total_er_pairs}")
print(f"Всего таблиц (ER пары × 2 + extra): {data_gen_config.total_tables}")
print(f"Output: {data_gen_config.output_dir}/")

In [ ]:
generator = UnifiedDatasetGenerator(data_gen_config)
generator.generate()

if generator.interrupted:
    print("⚠ Генерация была прервана! Данные могут быть неполными.")
else:
    print("✓ Генерация завершена успешно.")

## 1.3 Статистический анализ SM датасета

In [ ]:
# Загрузка SM датасета
sm_dataset_path = os.path.join(data_gen_config.output_dir, data_gen_config.sm_dataset_file)
sm_metadata_path = os.path.join(data_gen_config.output_dir, data_gen_config.sm_metadata_file)

with open(sm_dataset_path, 'r', encoding='utf-8') as f:
    sm_data = json.load(f)

with open(sm_metadata_path, 'r', encoding='utf-8') as f:
    sm_meta = json.load(f)

print(f"SM записей: {len(sm_data)}")
print(f"Уникальных base_name: {sm_meta['stats']['unique_base_names']}")
print(f"Таблиц из ER: {sm_meta['stats']['tables_from_er']}")
print(f"Таблиц extra SM: {sm_meta['stats']['tables_from_extra']}")
print(f"Размерность эмбеддинга: {len(sm_data[0]['embedding'])}")

In [ ]:
# Распределение классов (base_name)
class_counts = Counter(entry['base_name'] for entry in sm_data)
class_df = pd.DataFrame(
    sorted(class_counts.items(), key=lambda x: x[1], reverse=True),
    columns=['base_name', 'count']
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Барплот классов
axes[0].barh(class_df['base_name'], class_df['count'], color=sns.color_palette('viridis', len(class_df)))
axes[0].set_xlabel('Количество записей')
axes[0].set_title('Распределение классов (base_name)')
axes[0].invert_yaxis()

# Boxplot количества по классам
axes[1].boxplot(class_df['count'], vert=True)
axes[1].set_title('Статистика по количеству записей в классе')
axes[1].set_ylabel('Количество записей')

plt.tight_layout()
plt.show()

print(f"\nСтатистика по классам:")
print(f"  min = {class_df['count'].min()}, max = {class_df['count'].max()}, "
      f"mean = {class_df['count'].mean():.1f}, median = {class_df['count'].median():.1f}")
print(f"  std = {class_df['count'].std():.1f}")
print(f"  Коэффициент вариации = {class_df['count'].std() / class_df['count'].mean() * 100:.1f}%")

In [ ]:
# Распределение типов данных
dtype_counts = Counter(entry['data_type'] for entry in sm_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels, sizes = zip(*sorted(dtype_counts.items(), key=lambda x: x[1], reverse=True))
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Распределение типов данных')

# Распределение по источнику (er_train, er_val, er_test, sm_extra)
source_counts = Counter(entry['source'] for entry in sm_data)
labels_s, sizes_s = zip(*sorted(source_counts.items(), key=lambda x: x[1], reverse=True))
axes[1].pie(sizes_s, labels=labels_s, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Распределение по источнику')

plt.tight_layout()
plt.show()

In [ ]:
# Анализ длин описаний
desc_lengths = [len(entry['description']) for entry in sm_data]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(desc_lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Длина описания (символы)')
axes[0].set_ylabel('Количество')
axes[0].set_title('Распределение длин описаний столбцов')
axes[0].axvline(np.mean(desc_lengths), color='red', linestyle='--', label=f'mean={np.mean(desc_lengths):.0f}')
axes[0].legend()

# Длины описаний по классам
desc_by_class = {}
for entry in sm_data:
    desc_by_class.setdefault(entry['base_name'], []).append(len(entry['description']))

class_names = sorted(desc_by_class.keys())
axes[1].boxplot(
    [desc_by_class[c] for c in class_names],
    labels=class_names,
    vert=True
)
axes[1].set_title('Длины описаний по классам')
axes[1].set_ylabel('Длина описания')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Анализ эмбеддингов: нормы и распределение
embeddings = np.array([entry['embedding'] for entry in sm_data])
norms = np.linalg.norm(embeddings, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(norms, bins=50, color='coral', edgecolor='white')
axes[0].set_xlabel('L2 норма эмбеддинга')
axes[0].set_title('Распределение норм эмбеддингов')
axes[0].axvline(np.mean(norms), color='red', linestyle='--', label=f'mean={np.mean(norms):.2f}')
axes[0].legend()

# Средний эмбеддинг по компонентам (первые 50)
axes[1].bar(range(50), np.mean(embeddings[:, :50], axis=0), color='teal')
axes[1].set_xlabel('Компонента')
axes[1].set_title('Средние значения первых 50 компонент')

# Распределение значений эмбеддингов (сэмпл)
sample_flat = embeddings[:100].flatten()
axes[2].hist(sample_flat, bins=100, color='mediumpurple', edgecolor='white')
axes[2].set_xlabel('Значение компоненты')
axes[2].set_title('Распределение значений (100 эмбеддингов)')

plt.tight_layout()
plt.show()

print(f"Размерность: {embeddings.shape[1]}")
print(f"Нормы: min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}, std={norms.std():.4f}")

In [ ]:
# Матрица косинусных сходств средних эмбеддингов по классам
from sklearn.metrics.pairwise import cosine_similarity

class_embeddings = {}
for entry in sm_data:
    class_embeddings.setdefault(entry['base_name'], []).append(entry['embedding'])

class_names_sorted = sorted(class_embeddings.keys())
mean_embeddings = np.array([np.mean(class_embeddings[c], axis=0) for c in class_names_sorted])

sim_matrix = cosine_similarity(mean_embeddings)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(class_names_sorted)))
ax.set_yticks(range(len(class_names_sorted)))
ax.set_xticklabels(class_names_sorted, rotation=45, ha='right')
ax.set_yticklabels(class_names_sorted)
ax.set_title('Косинусное сходство средних эмбеддингов по классам (до обучения SM)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 1.4 Статистический анализ ER датасета

In [ ]:
# Анализ ER пар
er_raw_dir = os.path.join(data_gen_config.output_dir, data_gen_config.er_raw_dir)
er_stats = []

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(er_raw_dir, split)
    if not os.path.exists(split_dir):
        continue
    
    meta_files = sorted(Path(split_dir).glob('*_meta.json'))
    for meta_file in meta_files:
        with open(meta_file, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        meta['split'] = split
        er_stats.append(meta)

er_df = pd.DataFrame(er_stats)
print(f"Всего ER пар: {len(er_df)}")
print(f"\nПо сплитам:")
print(er_df.groupby('split')[['num_rows_a', 'num_rows_b', 'num_duplicates']].agg(['mean', 'min', 'max']).round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Распределение количества строк в таблицах
all_rows = list(er_df['num_rows_a']) + list(er_df['num_rows_b'])
axes[0].hist(all_rows, bins=range(min(all_rows), max(all_rows) + 2), 
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Количество строк')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение размеров таблиц (ER)')
axes[0].axvline(np.mean(all_rows), color='red', linestyle='--', label=f'mean={np.mean(all_rows):.1f}')
axes[0].legend()

# Распределение количества дубликатов
axes[1].hist(er_df['num_duplicates'], bins=range(0, er_df['num_duplicates'].max() + 2),
             color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Количество дубликатов')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение количества дубликатов в паре')
axes[1].axvline(er_df['num_duplicates'].mean(), color='red', linestyle='--', 
                label=f'mean={er_df["num_duplicates"].mean():.1f}')
axes[1].legend()

# Доля дубликатов
dup_ratio = er_df['num_duplicates'] / ((er_df['num_rows_a'] + er_df['num_rows_b']) / 2)
axes[2].hist(dup_ratio, bins=30, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Доля дубликатов')
axes[2].set_ylabel('Частота')
axes[2].set_title('Доля дубликатов от среднего размера таблицы')

plt.tight_layout()
plt.show()

In [ ]:
# Пример пары таблиц
example_split = 'train'
example_dir = os.path.join(er_raw_dir, example_split)
example_a = pd.read_csv(os.path.join(example_dir, 'pair_0000_table_a.csv'))
example_b = pd.read_csv(os.path.join(example_dir, 'pair_0000_table_b.csv'))

with open(os.path.join(example_dir, 'pair_0000_meta.json'), 'r', encoding='utf-8') as f:
    example_meta = json.load(f)

print(f"Таблица A: {example_a.shape}, столбцы: {list(example_a.columns)}")
print(f"Таблица B: {example_b.shape}, столбцы: {list(example_b.columns)}")
print(f"Дубликаты: {example_meta['num_duplicates']} пар")
print(f"\nColumn mapping A: {example_meta['column_mapping_a']}")
print(f"Column mapping B: {example_meta['column_mapping_b']}")

print("\n--- Таблица A (первые 5 строк) ---")
display(example_a.head())

print("\n--- Таблица B (первые 5 строк) ---")
display(example_b.head())

In [ ]:
# Анализ разнообразия имён столбцов для одного base_name
column_name_variants = {}
for entry in sm_data:
    column_name_variants.setdefault(entry['base_name'], set()).add(entry['column_name'])

variant_counts = {k: len(v) for k, v in column_name_variants.items()}
variant_df = pd.DataFrame(
    sorted(variant_counts.items(), key=lambda x: x[1], reverse=True),
    columns=['base_name', 'unique_display_names']
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(variant_df['base_name'], variant_df['unique_display_names'], color=sns.color_palette('husl', len(variant_df)))
ax.set_xlabel('Уникальных вариантов имён столбцов')
ax.set_title('Разнообразие имён столбцов по классам')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Примеры вариантов для нескольких классов
for base_name in list(column_name_variants.keys())[:5]:
    variants = sorted(column_name_variants[base_name])
    print(f"\n{base_name} ({len(variants)} вариантов): {variants[:10]}")

## 1.5 Визуализация эмбеддингов (t-SNE до обучения SM)

In [ ]:
from sklearn.manifold import TSNE

# t-SNE на сырых эмбеддингах (до обучения ProjectionHead)
embeddings = np.array([entry['embedding'] for entry in sm_data])
labels = [entry['base_name'] for entry in sm_data]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_result = tsne.fit_transform(embeddings)

unique_labels = sorted(set(labels))
cmap = plt.cm.get_cmap('tab20', len(unique_labels))

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    mask = np.array(labels) == label
    ax.scatter(tsne_result[mask, 0], tsne_result[mask, 1],
               c=[cmap(i)], label=label, alpha=0.6, s=20)

ax.set_title('t-SNE: сырые эмбеддинги столбцов (до обучения SM)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 1.6 Итоговая сводка

In [ ]:
print("=" * 60)
print("ИТОГОВАЯ СВОДКА ДАТАСЕТА")
print("=" * 60)
print(f"\nSchema Matching:")
print(f"  Записей: {len(sm_data)}")
print(f"  Классов: {sm_meta['stats']['unique_base_names']}")
print(f"  Размерность эмбеддинга: {embeddings.shape[1]}")
print(f"  Файл: {sm_dataset_path}")
print(f"\nEntity Resolution:")
print(f"  Пар таблиц: {len(er_df)}")
for split in ['train', 'val', 'test']:
    split_df = er_df[er_df['split'] == split]
    print(f"    {split}: {len(split_df)} пар, "
          f"avg rows={split_df['num_rows_a'].mean():.1f}/{split_df['num_rows_b'].mean():.1f}, "
          f"avg dups={split_df['num_duplicates'].mean():.1f}")
print(f"  Директория: {er_raw_dir}")
print(f"\nСледующий шаг: Pipeline 2 — Обучение Schema Matching")